In [23]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google API Key: ")

Enter your Google API Key: ··········


In [25]:
# --- Install ---
!pip install -q -U langchain langchain-core langchain-community langchain-google-genai langchain-text-splitters chromadb pypdf tiktoken

# --- Imports ---
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# --- Load PDF ---
loader = PyPDFLoader("/content/sorting algorithms.pdf")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
texts = text_splitter.split_documents(documents)

# --- Google Embeddings ---
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(texts, embedding=embeddings)
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 10})

# --- LLM (Gemini) ---
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.3)

# --- Retrieval Chain ---
system_prompt = """You are a helpful assistant for question answering using document context.
If unsure, say you don't know. Context:\n{context}"""
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{question}")
])

retrieval_chain = (
    {"context": retriever | (lambda docs: "\n\n".join([d.page_content for d in docs])),
     "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# ---  Example Query ---
response = retrieval_chain.invoke("Explain selection sort")
print(response)


Selection sort is an iterative sorting algorithm. It works by repeatedly finding the minimum element from the unsorted portion of an array and placing it at the beginning of the array. The array is divided into two sub-arrays: the sorted sub-array at the beginning and the unsorted sub-array at the end. The algorithm selects the smallest element from the unsorted sub-array and swaps it with the element in the current position. The sorting continues until the array is completely sorted.

Selection sort is not suitable for large-scale sorting tasks because its time complexity of O(n^2) leads to a significant increase in comparisons and swaps as the number of elements grows. Faster algorithms like quicksort or merge sort would be preferable for large collections.


In [26]:
# --- Example Query ---
user_query = "What is selection sort?"
response = retrieval_chain.invoke(user_query)
print(response)

Selection sort is a sorting algorithm that selects the smallest element from the unsorted sub-array and swaps it with the element in the current position. The sorting continues until the array is completely sorted. It is simple and easy to understand, and performs well with small datasets where it has a quadratic time complexity of O(n^2). However, it has a worst-case time complexity of O(n^2), and is not an adaptive sorting algorithm.
